# Lecture 21: Extremals without a boundary-value problem

**The challenge.** Every solve so far fixed both ends of a motion and
searched for the costates that join them. With limits, that search had a
second, harder part: guessing which arcs the answer rides, one violation at a
time, with every guess judged against end conditions a whole motion away
(Lectures 15 and 20). This lecture turns the problem around. Fix only the
start, pick *any* costates, and integrate: the extremal that comes out is
exact for the problem from that start to wherever it ends. Limits are then
handled one junction at a time, by conditions that are all local.

**After this lecture you can**

- say which conditions hold at the ends of a motion, and why, when both ends
  are fixed states, nothing has to be integrated backward;
- count the free parameters of an arc going forward (its jump) and going
  backward (its length), and say why the counts agree;
- explain why a junction is local when the end is free, and what that does to
  the search for the arcs;
- grow an arc from a contact, for an entry at a corner and for a tangential one;
- name the ways a draw fails, and why none of them produces a wrong extremal.

## 1. Which conditions hold at the ends?

The necessary conditions are a differential equation for the state and
costates together, plus conditions at the two ends. With a terminal cost
$\phi$ and terminal constraints $\psi(x(t_f)) = 0$ with multipliers $\nu$,

$$
\lambda(t_f) = \frac{\partial \phi}{\partial x} + \Big(\frac{\partial \psi}{\partial x}\Big)^{\!T} \nu,
\qquad
H(t_f) + \frac{\partial \phi}{\partial t} = 0 \ \text{ if } t_f \text{ is free}.
$$

A component of the final state that $\psi$ leaves free, with no terminal
cost on it, has $\lambda_i(t_f) = 0$. That is the familiar "costates hit
zero" of orbit raising, where the final angle is free: there the end has
known costates, and integrating backward from it is natural.

A component that $\psi$ fixes gets its own $\nu_i$, and its costate is free
at the end. The arm fixes the whole final state (position, rate and
acceleration of every joint), so *no* costate is known at $t_f$, and none
at $t_0$ either. What is left is $H = 0$ when the final time is free. That
is not a condition at $t_f$ alone: the problem does not depend on time, so
$H$ is constant along every extremal, and $H(t_0) = 0$ is the same
condition.

So: pick a start $x_0$, pick costates $\lambda_0$ on $H = 0$, and integrate
for any duration $T$. Whatever end $x(T)$ comes out, the trajectory
satisfies every condition of the problem from $x_0$ to $x(T)$. Direction is
a choice, not a requirement. Integrating backward from a chosen end is the
mirror image, and is the better choice only when the end, not the start, is
what you want to fix.

The example is one jerk-controlled axis in minimum time with acceleration
limits $|a| \le A$ (order 1 under jerk control), whose two ends are
parameters, so one set of kernels serves every start and every end.

In [ ]:
import warnings
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

from auto_optimal_control.embedding import Polisher
from auto_optimal_control.generation import ExtremalGenerator, Rejected
from auto_optimal_control.homotopy import Homotopy
from auto_optimal_control.policy import SecondaryCost
from auto_optimal_control.problem import Box, OptimalControlProblem, PathConstraint

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

t, x, v, a, j = sp.symbols("t x v a j", real=True)
J, A = sp.symbols("J A", positive=True)
e0, ef = sp.symbols("x0 v0 a0", real=True), sp.symbols("xf vf af", real=True)
axis = OptimalControlProblem(
    name="axis_between", time=t, states=[x, v, a], controls=[j], params=[*e0, *ef, J, A],
    dynamics=[v, a, j], running_cost=1, initial_state=dict(zip([x, v, a], e0)),
    final_state=dict(zip([x, v, a], ef)), control_sets=[Box(j, -J, J)],
    path_constraints=[PathConstraint(a - A, "a_max"), PathConstraint(-a - A, "a_min")])

def params(x0, xf):
    return [*x0, *xf, 1.0, 1.0]                  # J = A = 1

bang = Homotopy(axis, policy="as_stated")       # minimum time as stated: bang-bang jerk
gen = ExtremalGenerator.from_homotopy(bang, params(np.zeros(3), np.zeros(3)))

rng = np.random.default_rng(3)
lam0 = rng.normal(size=3)
free = gen.generate(np.zeros(3), lam0, 2.0)
print("arcs:", free.structure, " end:", np.round(free.xf, 4), " t_f =", free.tf)
print("H(t0) after scaling lam0 onto H = 0:", bang.exact_kernels.hamiltonian(0.0, np.r_[free.x0, free.lam0], gen.params))
print("spread of H along the motion:", free.checks["hamiltonian"])

The draw was scaled along its ray onto $H = 0$
(`ExtremalGenerator.on_level_set`: along $s\lambda_0$ the Hamiltonian
$\min_u L + s\,\lambda_0^T f$ is concave in $s$ and starts at
$\min_u L = 1$, so it has one positive root). $H$ then stays at zero to
rounding. To check that this really is the extremal from rest to that end,
solve that boundary-value problem with it as the guess: the solve should
accept it at once, with the same time.

In [ ]:
polish = Polisher(bang, params)
check = polish.solve(free.x0, free.xf, [free])
print("accepted:", check.accepted, " t_f:", check.tf, " difference:", abs(check.tf - free.tf))

## 2. A limit, one junction at a time

Arcs are what make this more than one integration. A trajectory from
arbitrary costates either stays inside its limits, and is then a valid
extremal as it stands, or it breaks one, and then it is not an extremal of
the constrained problem at all. The generator turns each violation into a
boundary arc and makes that arc's junction conditions hold. They are the
conditions of Lecture 14, but now none of them is coupled to an end
condition:

| | in a solve (both ends fixed) | in generation (end free) |
|---|---|---|
| unknowns of an arc | $t_\text{in}$, $t_\text{out}$, $\pi$ | $t_\text{in}$, $\pi$ |
| at the entry | $g = 0$ and tangency, or $H$ continuity | the same |
| at the exit | $\mu = 0$ (a row) | where $\mu$ reaches zero (an event) |
| what else the arc must satisfy | every end condition, through the rest of the motion | nothing |

Going forward, the jump $\pi$ at the entry is free: on the arc it is the
initial value of the multiplier, $\mu(t_\text{in}^+) = \pi$, and it decides
how long the arc lasts, since the arc ends where $\mu$ falls back to zero.
Going backward the roles swap: the arc's length is free and $\pi$ follows
from the junction. Either way an arc costs one condition at its entry and
gives back one free parameter, so the family of extremals keeps its
dimension: any start, any end.

This is why the search for the structure becomes easy. In a solve, a wrong
guess about an arc shows up only as a failure to meet the far end, and the
search has to guess again (Lecture 20's greedy route). Here an arc is right
the moment its own entry conditions hold; the end is whatever it is.

In [ ]:
def survey(h, n, seed, moving=False, **kwargs):
    g = ExtremalGenerator.from_homotopy(h, params(np.zeros(3), np.zeros(3)), **kwargs)
    rng = np.random.default_rng(seed)
    out, reasons = [], Counter()
    for _ in range(n):
        x0 = np.array([0.0, *rng.uniform(-0.5, 0.5, 2)]) if moving else np.zeros(3)
        try:
            e = g.generate(x0, rng.normal(size=3), rng.uniform(0.5, 6), rng=rng)
            out.append(e)
            reasons["truncated: " + str(e.checks["truncated"]) if e.checks.get("truncated")
                    else "complete"] += 1
        except Rejected as r:
            reasons["rejected: " + r.reason] += 1
    return g, out, reasons

g_bang, bang_set, why = survey(bang, 120, seed=0)
print(dict(why))
print(Counter(len(e.arcs) // 2 for e in bang_set), "(arcs per extremal: how many)")
print("worst junction residual", max(e.junction_residual for e in bang_set))
print("worst violation off the arcs", max(e.checks["violation"] for e in bang_set),
      " smallest mu on them", min(e.checks["multiplier_min"] for e in bang_set))

Every extremal kept meets its junction conditions to about $10^{-11}$ and
breaks no limit. Some draws were *truncated*: when an arc cannot be made
valid (Section 4), the motion is ended just before the violation instead.
Nothing before that point changes, so the shorter motion is still exact.
Here are a few with arcs.

In [ ]:
with_arcs = [e for e in bang_set if e.arcs][:4]
fig, axes = plt.subplots(2, 1, figsize=(7.5, 5), sharex=True)
for e, color in zip(with_arcs, [BLUE, ORANGE, GRAY, "#4c9a2a"]):
    shoot, vv = g_bang.shooting_of(e)
    tr = shoot.trajectory(vv, 1500)
    axes[0].plot(tr["t"], tr["z"][:, 2], color=color)
    axes[1].plot(tr["t"], tr["u"][:, 0], color=color)
    on = tr["active"].any(axis=1)
    axes[0].plot(tr["t"][on], tr["z"][on, 2], color=color, lw=4, alpha=0.4)
axes[0].axhline(1, color="k", lw=0.6, ls=":"); axes[0].axhline(-1, color="k", lw=0.6, ls=":")
axes[0].set_ylabel("acceleration a"); axes[1].set_ylabel("jerk j"); axes[1].set_xlabel("t")
axes[0].set_title("Generated extremals: arcs (thick) on a = ±A, entered at a corner")
fig.tight_layout()

*Each motion starts at rest and ends wherever its costates take it. The jerk
is bang-bang on the free arcs and zero on the arcs, so each arc is entered
at a corner of the acceleration.*

## 3. Two kinds of entry

How an arc is entered depends on the control law, not on the problem's
size, and the generator follows the problem's own law.

- **The control can jump** (its Hamiltonian is linear in it: minimum time as
  stated, an $L_1$ cost). At the entry the jerk jumps from $\pm J$ to the
  riding value, so the acceleration arrives at its limit at an angle. The
  entry condition is $H$ continuity, which here reads $S = \lambda_a = 0$ at
  the moment $a$ reaches $A$.
- **The control law is continuous** ($H$ strictly convex in it, clipped or
  not: the default policy's jerk cost). Then the control is continuous at
  a junction (Jacobson, Lele and Speyer, 1971), however tightly it is
  bounded, and the arrival is tangential: $g = 0$ and $\dot g = 0$
  together.

The second case is the one that needs care. A trajectory from arbitrary
costates either crosses its limit at an angle, which no extremal does, or
peaks just below it. So an arc is *born* at a peak. The generator raises
that limit to just below the peak (the limits' offsets, which the active-set
route also uses), where the contact is tangential with zero jump and nearly
satisfied already, and lowers the offset back to zero in adaptive steps,
projecting at each. When the free trajectory peaks exactly on the limit,
the jump $\pi$ is chosen for an arc of a sampled length, and the arc ends
where $\mu$ reaches zero.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    smooth = Homotopy(axis, policy=SecondaryCost())
g_smooth, smooth_set, why = survey(smooth, 80, seed=1)
print("regular control law:", g_smooth.regular)
print(dict(why))
k, p = g_smooth.kernels, g_smooth.params
rates = []
for e in smooth_set:
    shoot, vv = g_smooth.shooting_of(e)
    _, _, segs = shoot._evaluate(vv, jac=False)
    for (kind, c), (t_in, _, pi) in zip([q for q in e.arcs if q[0] == "enter"], e.values):
        before = next(s for s in segs if s[0] is None and abs(s[3] - t_in) < 1e-12)
        rates.append(abs(k.constraint_rate(t_in, before[-1]["z"], p)[c]))
print(f"{len(rates)} tangential entries; largest dg/dt arriving at them: {max(rates):.1e}")

## 4. When a draw cannot be made valid

Projection is a local correction, so it can fail. Three failures came up,
and each one says something about the problem rather than about the method.

1. **The limit cannot be ridden there.** Riding a torque limit fixes a
   combination of the jerks. Near a free trajectory's peak that combination
   may need a jerk beyond its bound: on the two-link arm of Lecture 19 one
   birth needed $j_2 = 10.55$ against $j_\max = 10$. No extremal with that
   arc exists nearby, and the projection is right to stall. The generator
   then ends the motion before the violation (*truncation*), which is still
   exact.
2. **The control is saturated all the way to the peak.** Then the
   trajectory up to the peak does not depend on the costates at all, so no
   change in $\lambda_0$ can move the peak. The start can: with
   `vary_start="positions"` (or `"all"`) the projection may also move the
   joint angles (or the whole start state), which moves the peak directly.
   Rest stays rest with `"positions"`.
3. **The limit is still being approached at the end.** Its peak, and so the
   arc, lies beyond $T$. The duration is free, so the motion is extended
   (up to three times the drawn $T$) until the peak is inside.

What never happens is a wrong extremal. Every kept sample is checked for the
conditions a solution must meet beyond its junction rows: no violation off
its arcs, $\mu \ge 0$ on them, $\pi \ge 0$, riding controls inside their
bounds, and $H = 0$ along the whole motion.

One more check is worth knowing about. An extremal is only a candidate for
a minimum until its *conjugate point*, where the map from costates to the
state reached first becomes singular (`ExtremalGenerator.conjugate_points`
tests the sign of that determinant along a free arc, the junction
conditions held). It is informative only while the control is not
saturated: a saturated control does not move with the costates at all, so
the determinant is zero there for a reason that has nothing to do with
optimality, and in the bang-bang case the test needs a different form.
That is the most important item still open in this lecture's method.

## 5. Anchored at a junction

The first two failures of Section 4 have one cause: the entry is *reached* by integrating
towards it, and near a saturated control the costates cannot steer the
path's peak onto the limit. Integration has a direction only because we gave
it one. So choose the entry itself and integrate away from it, both ways
(`ExtremalGenerator.generate_anchored`):

1. **Build the entry.** The conditions for an entry onto limit $c$ hold at
   one point $(x^*, \lambda^*)$: $g_c(x^*) = 0$, a tangential arrival, and
   $H = 0$. They need no integration. For a continuous control law the
   generator moves the state onto the limit, takes the control that rides it
   there (it must be strictly inside its bounds), and finds costates whose
   *free* control is that control, on $H = 0$. A free control that rides the
   limit has $\dot g = 0$, so the entry is tangential, and since nothing is
   saturated every step has a simple root.
2. **Backward to a start.** The free equations integrate backward from the
   entry; the start is wherever that ends, or just after a limit broken on
   the way, so the start is not chosen (moving starts, in general).
3. **Forward through the arc.** The jump $\pi$ is chosen for an arc of a
   sampled length, the arc ends where $\mu$ reaches zero, and the rest of the
   motion is grown as before.

The count still closes: 36 numbers at the entry less 3 conditions, plus
$\pi$ and the two durations, are 36, the same as a start and an end. On the
axis with the continuous law:

In [ ]:
def compare(g, n, seed):
    rng = np.random.default_rng(seed)
    out = {"plain": Counter(), "anchored": Counter()}
    for _ in range(n):
        x0 = np.array([0.0, *rng.uniform(-0.5, 0.5, 2)]); lam = rng.normal(size=3); T = rng.uniform(1, 5)
        for mode in out:
            try:
                e = (g.generate(x0, lam, T, rng=rng) if mode == "plain"
                     else g.generate_anchored(int(rng.integers(2)), np.r_[x0, lam], 0.4 * T, 0.6 * T, rng=rng))
                out[mode]["with arcs" if e.arcs else "no arcs"] += 1
            except Rejected:
                out[mode]["rejected"] += 1
    return out

for mode, c in compare(g_smooth, 40, seed=5).items():
    print(f"{mode:9s}", dict(c))

Anchored draws carry an arc by construction, and they are cheap: the entry
is a small algebraic solve and the rest two integrations. On the six-joint
arm, from the same kind of draws:

| 30 draws, random poses, small rates and accelerations | plain | anchored |
|---|---|---|
| kept | 30 (29 cut short before any arc) | 13 |
| with an arc | 1 | 12 (2 with two arcs) |
| seconds per draw | 20 | 6 |
| rejected: the tangency there is a trough of the torque, so the motion would arrive from outside | | 13 |
| rejected: propagation failed, or no entry could be built | | 4 |

Anchoring removes the failure it was built for: no entry fails to be
reached, and twelve times as many draws carry an arc, at a third of the
cost. Most of what is lost is lost at the anchor itself. At a tangential
point the torque either peaks (a valid entry) or dips (a contact from
outside), depending on the costates, and half the points built are dips.
That is the next thing to aim at: the costates keep freedom after the
entry conditions, enough to choose the sign of the torque's curvature.

## 6. Summary

- When both ends are fixed states, the costates are free at both ends, and
  $H = 0$ holds everywhere if it holds anywhere. Integrating from the start
  with any costates on $H = 0$ gives an exact extremal to wherever it ends.
- Going forward, an arc's jump is free and its exit is an event; going
  backward, its length is free. Either way the family keeps its dimension.
- Without an end condition to meet, every junction condition is local, so a
  wrong structure cannot hide behind the far end: an arc is valid when its
  own entry holds.
- The price is that the end is not chosen. Lecture 22 builds the map from
  many generated extremals that answers "from here to there".